In [1]:
import ee
import geemap
from utils.variables import (
    PROJECT,
    TEST_SITE_IDs,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    GLC_CLASSES,
    NFW_THRESHOLD,
    INTACTNESS_KERNEL_RADIUS
)

In [2]:
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [3]:
# Data imports
PAs = ee.FeatureCollection("WCMC/WDPA/current/polygons")
OECMs = ee.FeatureCollection("WCMC/WDOECM/current/polygons")
GLC = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
HGFC = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")
NFW = ee.ImageCollection("projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection")

In [4]:
# Get selected test sites
def get_test_sites():
    # Combine terrestrial PAs and OECMs into one collection
    all_sites = (ee.FeatureCollection([PAs, OECMs]).flatten()
             .filter(ee.Filter.eq("REALM", "Terrestrial")))
    # Select test sites by ID
    test_sites = (all_sites.filter(ee.Filter.inList("SITE_ID", TEST_SITE_IDs)))
    return test_sites

test_sites = get_test_sites()

In [5]:
# Process Global Land Cover Change data
def process_GLC():
    # Mosaic images in the collection that intersect test sites
    GLC_mosaic = GLC.filterBounds(test_sites).mosaic()
    # Select and rename bands corresponding to analysis period
    analysis_years = list[int](range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1))
    band_names = [f"b{year - 2000 + 1}" for year in analysis_years]
    new_band_names = [f"GLC_{year}" for year in analysis_years]
    GLC_selected = GLC_mosaic.select(band_names, new_band_names)
    # Remap class values to 1-36
    def remap_classes(band):
        return (GLC_selected.select(band)
                .remap(GLC_CLASSES, ee.List.sequence(1, len(GLC_CLASSES)), defaultValue=0)
                .rename([band]))
    remapped_bands = [remap_classes(band) for band in new_band_names]
    GLC_remapped = ee.Image.cat(remapped_bands)
    return GLC_remapped

GLC_processed = process_GLC()

In [6]:
# Process Global Pasture Watch data
def process_GPW():
    # Filter GPW collection to analysis period
    year_strings = [str(year) for year in range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1)]
    GPW_filtered = GPW.filter(ee.Filter.inList("system:index", year_strings)).toBands()
    GPW_renamed = GPW_filtered.rename([f"GPW_{year}" for year in year_strings])
    return GPW_renamed.unmask()

GPW_processed = process_GPW()

In [7]:
# Process Natural Forests of the World data
def process_NFW():
    # Mosaic images in the collection that intersect test sites
    NFW_mosaic = NFW.filterBounds(test_sites).mosaic()
    # Set probability threshold
    NFW_thresholded = NFW_mosaic.gte(NFW_THRESHOLD)
    return NFW_thresholded

NFW_processed = process_NFW()

In [8]:
# Process Hansen Global Forest Change data
def process_HGFC():
    # Select forest loss band
    HGFC_selected = HGFC.select("lossyear")
    # Mask to forest loss within analysis period
    analysis_mask = (HGFC_selected.gte(ANALYSIS_START_YR - 2000)
            .And(HGFC_selected.lte(ANALYSIS_END_YR - 2000)))
    HGFC_masked = HGFC_selected.updateMask(analysis_mask)
    return HGFC_masked.unmask()

HGFC_processed = process_HGFC()

In [9]:
# Create natural cover raster for ANALYSIS_END_YR by creating and inverting a non-natural mask
def get_natural_cover():
    # Select GLC data for ANALYSIS_END_YR
    GLC_current = GLC_processed.select(f"GLC_{ANALYSIS_END_YR}")
    # Select cropland (1-4) and impervious surfaces (30)
    anthro_classes = ee.List([1, 2, 3, 4, 30])
    anthro_mask = GLC_current.remap(anthro_classes, ee.List.repeat(1, anthro_classes.size()), defaultValue=0)
    # Select any forest loss pixels during the analysis period
    forest_loss_mask = HGFC_processed.gt(0)
    # Select any cultivated grassland pixels using GPW
    GPW_current = GPW_processed.select(f"GPW_{ANALYSIS_END_YR}")
    pasture_mask = GPW_current.eq(1)
    # Select any pixels of "Forest" class that do not coincide with Natural Forests
    forest_classes = ee.List([5,6,7,8,9,10,11,12,13,14])
    planted_forest_mask = (GLC_current.remap(forest_classes, ee.List.repeat(1, forest_classes.size()), defaultValue=0)
                           .updateMask(NFW_processed.eq(0))).unmask(0)
    # Combine all non-natural masks and invert; apply mask to landcover raster
    non_natural_mask = anthro_mask.add(forest_loss_mask).add(pasture_mask).add(planted_forest_mask)
    natural_cover_mask = non_natural_mask.eq(0)
    natural_cover = GLC_current.updateMask(natural_cover_mask)
    return natural_cover

natural_cover = get_natural_cover()

In [35]:
# Select site
def get_site_geom(site_id):
    return test_sites.filter(ee.Filter.eq("SITE_ID", site_id)).geometry()

site_geom = get_site_geom(555752316)

In [36]:
# Calculate percent natural cover for ANALYSIS_END_YR within a PA
def calc_pct_natural_cover(site_geom):
    # Calculate total area of site
    site_area = site_geom.area().getInfo()
    # Calculate total area of natural cover within site
    natural_cover_area = (ee.Image.pixelArea()
        .updateMask(natural_cover)
        .reduceRegion(ee.Reducer.sum(), site_geom, scale=30, maxPixels=1e13)
        .get('area')
        .getInfo())
    pct_natural_cover = (natural_cover_area / site_area) * 100
    return pct_natural_cover

pct_natural_cover = calc_pct_natural_cover(site_geom)
print(pct_natural_cover)

99.96358497017468


In [ ]:
# Create edge distance layer (distance to nearest non-habitat pixel)
def get_edge_distance(site_geom):
    # Buffer site to maximum search radius
    site_buffered = site_geom.buffer(INTACTNESS_KERNEL_RADIUS)
    # Create and invert habitat binary (habitat = 0, non-habitat = 1)
    habitat_binary = natural_cover.clip(site_buffered).unmask().eq(0)
    # Calculate distance to nearest non-habitat pixel
    edge_distance = habitat_binary.distance(ee.Kernel.euclidean(INTACTNESS_KERNEL_RADIUS, units='meters'))
    # Cap distance at maximum search radius
    edge_distance_cleaned = edge_distance.min(INTACTNESS_KERNEL_RADIUS).unmask(INTACTNESS_KERNEL_RADIUS).clip(site_geom)
    return edge_distance_cleaned

edge_distance = get_edge_distance(site_geom)

# TO DO: 
# - Morphologically close (dilate then erode) the habitat binary to close small gaps
# - Investigate if it's faster to limit the computation to pixels within 500m of non-habitat (start with non-habitat and work outwards)

In [ ]:
# Visualization

from utils.variables import GLC_PALETTE
Map = geemap.Map()
Map.add_basemap('Esri.WorldImagery')

# Map.addLayer(GLC_processed.select('GLC_2022'), {'min':0, 'max':36, 'palette': GLC_PALETTE}, "GLC")
# Map.addLayer(GPW_processed.select('GPW_2022').selfMask(), {'min':1, 'max':2, 'palette': ['#ffcd73','#ff9916']}, "GPW")
# Map.addLayer(NFW_processed.selfMask(), {'palette': 'teal'}, "NFW")
# Map.addLayer(HGFC_processed.selfMask(), {'min':ANALYSIS_START_YR-2000, 'max':ANALYSIS_END_YR-2000, 'palette': ['yellow', 'red']}, "HGFC")
# Map.addLayer(test_sites, {"color": "red"}, "Test sites")

# Map.addLayer(natural_cover, {'palette': GLC_PALETTE}, "Natural cover")

Map.addLayer(edge_distance.selfMask(), {'min':0, 'max':500, 'palette': ['red', 'white']}, 'Edge distance')

Map.addLayer(site_geom, {"color": "red"}, "Test site")
Map.centerObject(site_geom)

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…